# 03. Conditional Random Fields (CRF)

**Goal:** Train a sequence model that learns not only from our contextual features, but also learns the *transition probabilities* between tags (e.g., I-ORG is highly likely to follow B-ORG, but impossible to follow B-LOC).

In [1]:
import joblib
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from seqeval.metrics import classification_report, f1_score
import matplotlib.pyplot as plt

## 1. Load Data

In [2]:
X_train, y_train, X_test, y_test = joblib.load('../data/processed/crf_data.pkl')
print(f'Loaded {len(X_train)} train sentences and {len(X_test)} test sentences.')

Loaded 14041 train sentences and 3453 test sentences.


> **📌 Decision Note — Why Model Selection for NER?**
>
> **Chosen approach:** Conditional Random Fields (CRF)
>
> **Why this works:** CRFs explicitly model the transition probabilities between output labels in a sequence. It 'learns the grammar' of tags (e.g. I-PER must follow B-PER).
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | Hidden Markov Models (HMM) | Generative sequence model | Cannot handle rich, overlapping, contextual features easily. Much less expressive than CRFs. |
> | Random Forest | Great tabular model | Ignores the sequence of tags entirely! It predicts token $i$ without knowing what it predicted for token $i-1$. |
>
> **Why we chose this over alternatives:** CRFs are the absolute pinnacle of Classical NLP for sequence tagging because they combine rich feature engineering with sequential label awareness.

## 2. Train CRF

In [3]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # L1 regularization
    c2=0.1,  # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

print('Training CRF model (this takes ~15 seconds on M4 Macs)...')
crf.fit(X_train, y_train)
print('Done!')

Training CRF model (this takes ~15 seconds on M4 Macs)...


Done!


## 3. Evaluation
We use `seqeval` because standard classification metrics penalize NER systems incorrectly if they get the boundary slightly wrong. `seqeval` evaluates on the *Entity* level, not the token level.

In [4]:
y_pred = crf.predict(X_test)
print(classification_report(y_test, y_pred))

print(f'Entity-Level F1 Score: {f1_score(y_test, y_pred):.3f}')

              precision    recall  f1-score   support

         LOC       0.85      0.81      0.83      1668
        MISC       0.80      0.74      0.77       702
         ORG       0.74      0.70      0.72      1661
         PER       0.82      0.85      0.83      1617

   micro avg       0.80      0.78      0.79      5648
   macro avg       0.80      0.77      0.79      5648
weighted avg       0.80      0.78      0.79      5648

Entity-Level F1 Score: 0.790


## 4. Inspect Learned Transitions
Let's see what the CRF learned about the 'grammar' of tags!

In [5]:
from collections import Counter
def print_transitions(trans_features):
    for (label_from, label_to), weight in trans_features:
        print(f'{label_from:9} -> {label_to:9} : {weight:.4f}')

print('Top 5 Likely Transitions:')
print_transitions(Counter(crf.transition_features_).most_common(5))
print('\nTop 5 Unlikely Transitions:')
print_transitions(Counter(crf.transition_features_).most_common()[-5:])

Top 5 Likely Transitions:
B-PER     -> I-PER     : 5.0055
B-LOC     -> I-LOC     : 4.8999
I-MISC    -> I-MISC    : 4.8198
B-MISC    -> I-MISC    : 4.6081
I-LOC     -> I-LOC     : 4.4117

Top 5 Unlikely Transitions:
B-LOC     -> I-ORG     : -4.2529
B-PER     -> B-PER     : -4.5277
O         -> I-LOC     : -4.7248
O         -> I-MISC    : -4.9737
O         -> I-ORG     : -5.8372


## Key Takeaways
- [x] In Sequence Tagging, `X` is a sequence of tokens and `y` is a sequence of tags.
- [x] We engineer features by creating **Context Windows** around every single word.
- [x] **CRFs** are the classical masters of this task because they learn both the features of the word AND the transition grammar of the tags!